<a href="https://colab.research.google.com/github/naotarokkwjwj/soccer/blob/main/%E5%B9%B3%E5%9D%87%E5%89%8D%E3%81%AE%E3%83%87%E3%83%BC%E3%82%BF%E6%A8%AA%E6%B5%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. 分析したいチーム名をここで指定してください
# ------------------------------------------------------------------
target_team_name ="横浜Ｆ・マリノス"  # ← ここを '川崎フロンターレ' や 'アルビレックス新潟' などに変更可能

# ------------------------------------------------------------------
# 2. データ準備 (play.csv読み込み & トラッキングデータ擬似生成)
# ------------------------------------------------------------------
try:
    # play.csv 読み込み
    raw_play = pd.read_csv('play.csv', encoding='utf-8-sig')

    # 列名変更 (日本語対応)
    col_map = {
        'チーム名': 'Team', 'アクション名': 'Action',
        '攻撃履歴No': 'AttackID', 'ボールＸ': 'x', 'ボールＹ': 'y',
        'ハーフ開始相対時間': 'Time', '履歴No': 'HistoryNo', '攻撃方向': 'Dir',
        '選手名': 'Player'
    }
    df = raw_play.rename(columns={k: v for k, v in col_map.items() if k in raw_play.columns})

    # 座標変換 & Frame生成
    df['x_m'] = pd.to_numeric(df['x'], errors='coerce') / 3.0
    df['y_m'] = pd.to_numeric(df['y'], errors='coerce') / 3.0
    time_col = df.get('Time', df.get('HistoryNo'))
    df['Frame'] = (time_col * 25 + 100000).astype(int)

    # トラッキングデータ(擬似)生成
    target_frames = df['Frame'].unique()
    track_records = []
    # ※本来は本物のトラッキングデータを使いますが、ここではデモ用に生成します
    for f in target_frames:
        for i in range(1, 12):
            track_records.append({'Frame': f, 'No': i, 'X': np.random.randint(-4000, 4000), 'Y': np.random.randint(-3000, 3000)})
    df_track = pd.DataFrame(track_records)

    # ------------------------------------------------------------------
    # 3. 個別分析ロジック実行
    # ------------------------------------------------------------------
    print(f"\n【分析対象チーム: {target_team_name}】\n")

    # チーム絞り込み
    team_df = df[df['Team'] == target_team_name].copy()
    if team_df.empty:
        print("エラー: 指定されたチームのデータが見つかりません。")
    else:
        # シュートイベント抽出
        shots = team_df[team_df['Action'].str.contains('シュート|ゴール', na=False)]
        print(f"分析対象シュート数: {len(shots)} シーン\n")

        print("-" * 90)
        print(f"{'AttackID':<10} | {'Shooter':<12} | {'Order':<6} | {'Passer':<12} | {'Type':<10} | {'V_Pack':<6} | {'D_Out':<6}")
        print("-" * 90)

        # 各シュートについてループ
        for idx, shot in shots.iterrows():
            attack_id = shot['AttackID']
            sequence = team_df[team_df['AttackID'] == attack_id].sort_values('Frame')
            try:
                shot_seq_idx = sequence.index.get_loc(idx)
            except: continue

            # シュートシーンごとのデータを格納するリスト
            scene_passes = []
            pass_count = 0

            # 遡ってパスを解析
            for i in range(1, 20):
                if shot_seq_idx - i < 0: break
                if pass_count >= 5: break

                event = sequence.iloc[shot_seq_idx - i]

                if 'パス' in str(event['Action']) or 'クロス' in str(event['Action']):
                    pass_count += 1

                    # パッキング計算
                    start_x = event['x_m']
                    next_event = sequence.iloc[shot_seq_idx - i + 1]
                    end_x = next_event['x_m']
                    if pd.isna(end_x): continue
                    if event['Dir'] == 2: start_x, end_x = -start_x, -end_x

                    frame = event['Frame']
                    opponents = df_track[df_track['Frame'] == frame]
                    opp_x = opponents['X'] / 100.0

                    v_packed = 0
                    if end_x > start_x:
                        v_packed = opponents[(opp_x > start_x) & (opp_x < end_x)].shape[0]
                    d_outplayed = opponents[opp_x < end_x].shape[0]

                    dx = end_x - start_x
                    dy = abs(next_event['y_m'] - event['y_m'])
                    p_type = 'Vertical' if (dx > 5 and dx > dy) else ('Horizontal' if (dy > dx and dy > 5) else 'Link-up')

                    # 結果をリストに追加
                    scene_passes.append({
                        'Order': -pass_count,
                        'Passer': event['Player'],
                        'Type': p_type,
                        'V_Pack': v_packed,
                        'D_Out': d_outplayed
                    })

            # 順番を整えて表示 (-5 -> -1)
            for p in sorted(scene_passes, key=lambda x: x['Order']):
                # パスの種類を日本語化して見やすく
                type_str = "縦パス" if p['Type'] == 'Vertical' else ("横パス" if p['Type'] == 'Horizontal' else "繋ぎ")

                print(f"{attack_id:<10} | {str(shot['Player']):<12} | {p['Order']:<6} | {str(p['Passer']):<12} | {type_str:<10} | {p['V_Pack']:<6} | {p['D_Out']:<6}")

            # シーンごとの区切り線
            if len(scene_passes) > 0:
                print("-" * 90)

except Exception as e:
    print(f"エラーが発生しました: {e}")


【分析対象チーム: 横浜Ｆ・マリノス】

分析対象シュート数: 7 シーン

------------------------------------------------------------------------------------------
AttackID   | Shooter      | Order  | Passer       | Type       | V_Pack | D_Out 
------------------------------------------------------------------------------------------
4          | 渡辺　皓太        | -3     | エウベル         | 繋ぎ         | 0      | 10    
4          | 渡辺　皓太        | -2     | 永戸　勝也        | 繋ぎ         | 0      | 11    
4          | 渡辺　皓太        | -1     | エウベル         | 横パス        | 0      | 11    
------------------------------------------------------------------------------------------
42         | ヤン　マテウス      | -3     | エウベル         | 縦パス        | 4      | 11    
42         | ヤン　マテウス      | -2     | ヤン　マテウス      | 繋ぎ         | 0      | 9     
42         | ヤン　マテウス      | -1     | エウベル         | 縦パス        | 1      | 11    
------------------------------------------------------------------------------------------
81         | 井上　健太        | -

In [2]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. 分析したいチーム名をここで指定してください
# ------------------------------------------------------------------
target_team_name ="横浜Ｆ・マリノス"  # ← ここを '川崎フロンターレ' や 'アルビレックス新潟' などに変更可能

# ------------------------------------------------------------------
# 2. データ準備 (play.csv読み込み & トラッキングデータ擬似生成)
# ------------------------------------------------------------------
try:
    # play.csv 読み込み
    raw_play = pd.read_csv('play.csv', encoding='utf-8-sig')

    # 列名変更 (日本語対応)
    col_map = {
        'チーム名': 'Team', 'アクション名': 'Action',
        '攻撃履歴No': 'AttackID', 'ボールＸ': 'x', 'ボールＹ': 'y',
        'ハーフ開始相対時間': 'Time', '履歴No': 'HistoryNo', '攻撃方向': 'Dir',
        '選手名': 'Player'
    }
    df = raw_play.rename(columns={k: v for k, v in col_map.items() if k in raw_play.columns})

    # 座標変換 & Frame生成
    df['x_m'] = pd.to_numeric(df['x'], errors='coerce') / 3.0
    df['y_m'] = pd.to_numeric(df['y'], errors='coerce') / 3.0
    time_col = df.get('Time', df.get('HistoryNo'))
    df['Frame'] = (time_col * 25 + 100000).astype(int)

    # トラッキングデータ(擬似)生成
    target_frames = df['Frame'].unique()
    track_records = []
    # ※本来は本物のトラッキングデータを使いますが、ここではデモ用に生成します
    for f in target_frames:
        for i in range(1, 12):
            track_records.append({'Frame': f, 'No': i, 'X': np.random.randint(-4000, 4000), 'Y': np.random.randint(-3000, 3000)})
    df_track = pd.DataFrame(track_records)

    # ------------------------------------------------------------------
    # 3. 個別分析ロジック実行
    # ------------------------------------------------------------------
    print(f"\n【分析対象チーム: {target_team_name}】\n")

    # チーム絞り込み
    team_df = df[df['Team'] == target_team_name].copy()
    if team_df.empty:
        print("エラー: 指定されたチームのデータが見つかりません。")
    else:
        # シュートイベント抽出
        shots = team_df[team_df['Action'].str.contains('シュート|ゴール', na=False)]
        print(f"分析対象シュート数: {len(shots)} シーン\n")

        print("-" * 90)
        print(f"{'AttackID':<10} | {'Shooter':<12} | {'Order':<6} | {'Passer':<12} | {'Type':<10} | {'V_Pack':<6} | {'D_Out':<6}")
        print("-" * 90)

        # 各シュートについてループ
        for idx, shot in shots.iterrows():
            attack_id = shot['AttackID']
            sequence = team_df[team_df['AttackID'] == attack_id].sort_values('Frame')
            try:
                shot_seq_idx = sequence.index.get_loc(idx)
            except: continue

            # シュートシーンごとのデータを格納するリスト
            scene_passes = []
            pass_count = 0

            # 遡ってパスを解析
            for i in range(1, 20):
                if shot_seq_idx - i < 0: break
                if pass_count >= 5: break

                event = sequence.iloc[shot_seq_idx - i]

                if 'パス' in str(event['Action']) or 'クロス' in str(event['Action']):
                    pass_count += 1

                    # パッキング計算
                    start_x = event['x_m']
                    next_event = sequence.iloc[shot_seq_idx - i + 1]
                    end_x = next_event['x_m']
                    if pd.isna(end_x): continue
                    if event['Dir'] == 2: start_x, end_x = -start_x, -end_x

                    frame = event['Frame']
                    opponents = df_track[df_track['Frame'] == frame]
                    opp_x = opponents['X'] / 100.0

                    v_packed = 0
                    if end_x > start_x:
                        v_packed = opponents[(opp_x > start_x) & (opp_x < end_x)].shape[0]
                    d_outplayed = opponents[opp_x < end_x].shape[0]

                    dx = end_x - start_x
                    dy = abs(next_event['y_m'] - event['y_m'])
                    p_type = 'Vertical' if (dx > 5 and dx > dy) else ('Horizontal' if (dy > dx and dy > 5) else 'Link-up')

                    # 結果をリストに追加
                    scene_passes.append({
                        'Order': -pass_count,
                        'Passer': event['Player'],
                        'Type': p_type,
                        'V_Pack': v_packed,
                        'D_Out': d_outplayed
                    })

            # 順番を整えて表示 (-5 -> -1)
            for p in sorted(scene_passes, key=lambda x: x['Order']):
                # パスの種類を日本語化して見やすく
                type_str = "縦パス" if p['Type'] == 'Vertical' else ("横パス" if p['Type'] == 'Horizontal' else "繋ぎ")

                print(f"{attack_id:<10} | {str(shot['Player']):<12} | {p['Order']:<6} | {str(p['Passer']):<12} | {type_str:<10} | {p['V_Pack']:<6} | {p['D_Out']:<6}")

            # シーンごとの区切り線
            if len(scene_passes) > 0:
                print("-" * 90)

except Exception as e:
    print(f"エラーが発生しました: {e}")


【分析対象チーム: 横浜Ｆ・マリノス】

分析対象シュート数: 14 シーン

------------------------------------------------------------------------------------------
AttackID   | Shooter      | Order  | Passer       | Type       | V_Pack | D_Out 
------------------------------------------------------------------------------------------
51         | 井上　健太        | -5     | 上島　拓巳        | 縦パス        | 1      | 7     
51         | 井上　健太        | -4     | 天野　純         | 横パス        | 0      | 8     
51         | 井上　健太        | -3     | 渡辺　皓太        | 横パス        | 2      | 5     
51         | 井上　健太        | -2     | 小池　龍太        | 横パス        | 0      | 10    
51         | 井上　健太        | -1     | 渡辺　皓太        | 横パス        | 3      | 9     
------------------------------------------------------------------------------------------
53         | 西村　拓真        | -3     | 小池　龍太        | 横パス        | 0      | 5     
53         | 西村　拓真        | -2     | 上島　拓巳        | 横パス        | 0      | 7     
53         | 西村　拓真        | -1     | 小

In [3]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. 分析したいチーム名をここで指定してください
# ------------------------------------------------------------------
target_team_name ="横浜Ｆ・マリノス"  # ← ここを '川崎フロンターレ' や 'アルビレックス新潟' などに変更可能

# ------------------------------------------------------------------
# 2. データ準備 (play.csv読み込み & トラッキングデータ擬似生成)
# ------------------------------------------------------------------
try:
    # play.csv 読み込み
    raw_play = pd.read_csv('play.csv', encoding='utf-8-sig')

    # 列名変更 (日本語対応)
    col_map = {
        'チーム名': 'Team', 'アクション名': 'Action',
        '攻撃履歴No': 'AttackID', 'ボールＸ': 'x', 'ボールＹ': 'y',
        'ハーフ開始相対時間': 'Time', '履歴No': 'HistoryNo', '攻撃方向': 'Dir',
        '選手名': 'Player'
    }
    df = raw_play.rename(columns={k: v for k, v in col_map.items() if k in raw_play.columns})

    # 座標変換 & Frame生成
    df['x_m'] = pd.to_numeric(df['x'], errors='coerce') / 3.0
    df['y_m'] = pd.to_numeric(df['y'], errors='coerce') / 3.0
    time_col = df.get('Time', df.get('HistoryNo'))
    df['Frame'] = (time_col * 25 + 100000).astype(int)

    # トラッキングデータ(擬似)生成
    target_frames = df['Frame'].unique()
    track_records = []
    # ※本来は本物のトラッキングデータを使いますが、ここではデモ用に生成します
    for f in target_frames:
        for i in range(1, 12):
            track_records.append({'Frame': f, 'No': i, 'X': np.random.randint(-4000, 4000), 'Y': np.random.randint(-3000, 3000)})
    df_track = pd.DataFrame(track_records)

    # ------------------------------------------------------------------
    # 3. 個別分析ロジック実行
    # ------------------------------------------------------------------
    print(f"\n【分析対象チーム: {target_team_name}】\n")

    # チーム絞り込み
    team_df = df[df['Team'] == target_team_name].copy()
    if team_df.empty:
        print("エラー: 指定されたチームのデータが見つかりません。")
    else:
        # シュートイベント抽出
        shots = team_df[team_df['Action'].str.contains('シュート|ゴール', na=False)]
        print(f"分析対象シュート数: {len(shots)} シーン\n")

        print("-" * 90)
        print(f"{'AttackID':<10} | {'Shooter':<12} | {'Order':<6} | {'Passer':<12} | {'Type':<10} | {'V_Pack':<6} | {'D_Out':<6}")
        print("-" * 90)

        # 各シュートについてループ
        for idx, shot in shots.iterrows():
            attack_id = shot['AttackID']
            sequence = team_df[team_df['AttackID'] == attack_id].sort_values('Frame')
            try:
                shot_seq_idx = sequence.index.get_loc(idx)
            except: continue

            # シュートシーンごとのデータを格納するリスト
            scene_passes = []
            pass_count = 0

            # 遡ってパスを解析
            for i in range(1, 20):
                if shot_seq_idx - i < 0: break
                if pass_count >= 5: break

                event = sequence.iloc[shot_seq_idx - i]

                if 'パス' in str(event['Action']) or 'クロス' in str(event['Action']):
                    pass_count += 1

                    # パッキング計算
                    start_x = event['x_m']
                    next_event = sequence.iloc[shot_seq_idx - i + 1]
                    end_x = next_event['x_m']
                    if pd.isna(end_x): continue
                    if event['Dir'] == 2: start_x, end_x = -start_x, -end_x

                    frame = event['Frame']
                    opponents = df_track[df_track['Frame'] == frame]
                    opp_x = opponents['X'] / 100.0

                    v_packed = 0
                    if end_x > start_x:
                        v_packed = opponents[(opp_x > start_x) & (opp_x < end_x)].shape[0]
                    d_outplayed = opponents[opp_x < end_x].shape[0]

                    dx = end_x - start_x
                    dy = abs(next_event['y_m'] - event['y_m'])
                    p_type = 'Vertical' if (dx > 5 and dx > dy) else ('Horizontal' if (dy > dx and dy > 5) else 'Link-up')

                    # 結果をリストに追加
                    scene_passes.append({
                        'Order': -pass_count,
                        'Passer': event['Player'],
                        'Type': p_type,
                        'V_Pack': v_packed,
                        'D_Out': d_outplayed
                    })

            # 順番を整えて表示 (-5 -> -1)
            for p in sorted(scene_passes, key=lambda x: x['Order']):
                # パスの種類を日本語化して見やすく
                type_str = "縦パス" if p['Type'] == 'Vertical' else ("横パス" if p['Type'] == 'Horizontal' else "繋ぎ")

                print(f"{attack_id:<10} | {str(shot['Player']):<12} | {p['Order']:<6} | {str(p['Passer']):<12} | {type_str:<10} | {p['V_Pack']:<6} | {p['D_Out']:<6}")

            # シーンごとの区切り線
            if len(scene_passes) > 0:
                print("-" * 90)

except Exception as e:
    print(f"エラーが発生しました: {e}")


【分析対象チーム: 横浜Ｆ・マリノス】

分析対象シュート数: 13 シーン

------------------------------------------------------------------------------------------
AttackID   | Shooter      | Order  | Passer       | Type       | V_Pack | D_Out 
------------------------------------------------------------------------------------------
12         | 小池　龍太        | -5     | 渡辺　皓太        | 横パス        | 2      | 4     
12         | 小池　龍太        | -4     | 永戸　勝也        | 横パス        | 0      | 3     
12         | 小池　龍太        | -3     | エウベル         | 横パス        | 0      | 3     
12         | 小池　龍太        | -2     | 畠中　槙之輔       | 縦パス        | 1      | 8     
12         | 小池　龍太        | -1     | ヤン　マテウス      | 縦パス        | 1      | 11    
------------------------------------------------------------------------------------------
34         | アンデルソン　ロペス   | -5     | アンデルソン　ロペス   | 横パス        | 0      | 8     
34         | アンデルソン　ロペス   | -4     | ヤン　マテウス      | 縦パス        | 3      | 9     
34         | アンデルソン　ロペス   | -3     | 西

In [4]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. 分析したいチーム名をここで指定してください
# ------------------------------------------------------------------
target_team_name ="横浜Ｆ・マリノス"  # ← ここを '川崎フロンターレ' や 'アルビレックス新潟' などに変更可能

# ------------------------------------------------------------------
# 2. データ準備 (play.csv読み込み & トラッキングデータ擬似生成)
# ------------------------------------------------------------------
try:
    # play.csv 読み込み
    raw_play = pd.read_csv('play.csv', encoding='utf-8-sig')

    # 列名変更 (日本語対応)
    col_map = {
        'チーム名': 'Team', 'アクション名': 'Action',
        '攻撃履歴No': 'AttackID', 'ボールＸ': 'x', 'ボールＹ': 'y',
        'ハーフ開始相対時間': 'Time', '履歴No': 'HistoryNo', '攻撃方向': 'Dir',
        '選手名': 'Player'
    }
    df = raw_play.rename(columns={k: v for k, v in col_map.items() if k in raw_play.columns})

    # 座標変換 & Frame生成
    df['x_m'] = pd.to_numeric(df['x'], errors='coerce') / 3.0
    df['y_m'] = pd.to_numeric(df['y'], errors='coerce') / 3.0
    time_col = df.get('Time', df.get('HistoryNo'))
    df['Frame'] = (time_col * 25 + 100000).astype(int)

    # トラッキングデータ(擬似)生成
    target_frames = df['Frame'].unique()
    track_records = []
    # ※本来は本物のトラッキングデータを使いますが、ここではデモ用に生成します
    for f in target_frames:
        for i in range(1, 12):
            track_records.append({'Frame': f, 'No': i, 'X': np.random.randint(-4000, 4000), 'Y': np.random.randint(-3000, 3000)})
    df_track = pd.DataFrame(track_records)

    # ------------------------------------------------------------------
    # 3. 個別分析ロジック実行
    # ------------------------------------------------------------------
    print(f"\n【分析対象チーム: {target_team_name}】\n")

    # チーム絞り込み
    team_df = df[df['Team'] == target_team_name].copy()
    if team_df.empty:
        print("エラー: 指定されたチームのデータが見つかりません。")
    else:
        # シュートイベント抽出
        shots = team_df[team_df['Action'].str.contains('シュート|ゴール', na=False)]
        print(f"分析対象シュート数: {len(shots)} シーン\n")

        print("-" * 90)
        print(f"{'AttackID':<10} | {'Shooter':<12} | {'Order':<6} | {'Passer':<12} | {'Type':<10} | {'V_Pack':<6} | {'D_Out':<6}")
        print("-" * 90)

        # 各シュートについてループ
        for idx, shot in shots.iterrows():
            attack_id = shot['AttackID']
            sequence = team_df[team_df['AttackID'] == attack_id].sort_values('Frame')
            try:
                shot_seq_idx = sequence.index.get_loc(idx)
            except: continue

            # シュートシーンごとのデータを格納するリスト
            scene_passes = []
            pass_count = 0

            # 遡ってパスを解析
            for i in range(1, 20):
                if shot_seq_idx - i < 0: break
                if pass_count >= 5: break

                event = sequence.iloc[shot_seq_idx - i]

                if 'パス' in str(event['Action']) or 'クロス' in str(event['Action']):
                    pass_count += 1

                    # パッキング計算
                    start_x = event['x_m']
                    next_event = sequence.iloc[shot_seq_idx - i + 1]
                    end_x = next_event['x_m']
                    if pd.isna(end_x): continue
                    if event['Dir'] == 2: start_x, end_x = -start_x, -end_x

                    frame = event['Frame']
                    opponents = df_track[df_track['Frame'] == frame]
                    opp_x = opponents['X'] / 100.0

                    v_packed = 0
                    if end_x > start_x:
                        v_packed = opponents[(opp_x > start_x) & (opp_x < end_x)].shape[0]
                    d_outplayed = opponents[opp_x < end_x].shape[0]

                    dx = end_x - start_x
                    dy = abs(next_event['y_m'] - event['y_m'])
                    p_type = 'Vertical' if (dx > 5 and dx > dy) else ('Horizontal' if (dy > dx and dy > 5) else 'Link-up')

                    # 結果をリストに追加
                    scene_passes.append({
                        'Order': -pass_count,
                        'Passer': event['Player'],
                        'Type': p_type,
                        'V_Pack': v_packed,
                        'D_Out': d_outplayed
                    })

            # 順番を整えて表示 (-5 -> -1)
            for p in sorted(scene_passes, key=lambda x: x['Order']):
                # パスの種類を日本語化して見やすく
                type_str = "縦パス" if p['Type'] == 'Vertical' else ("横パス" if p['Type'] == 'Horizontal' else "繋ぎ")

                print(f"{attack_id:<10} | {str(shot['Player']):<12} | {p['Order']:<6} | {str(p['Passer']):<12} | {type_str:<10} | {p['V_Pack']:<6} | {p['D_Out']:<6}")

            # シーンごとの区切り線
            if len(scene_passes) > 0:
                print("-" * 90)

except Exception as e:
    print(f"エラーが発生しました: {e}")


【分析対象チーム: 横浜Ｆ・マリノス】

分析対象シュート数: 20 シーン

------------------------------------------------------------------------------------------
AttackID   | Shooter      | Order  | Passer       | Type       | V_Pack | D_Out 
------------------------------------------------------------------------------------------
2          | 植中　朝日        | -5     | ポープ　ウィリアム    | 横パス        | 1      | 1     
2          | 植中　朝日        | -4     | 渡邊　泰基        | 縦パス        | 3      | 3     
2          | 植中　朝日        | -3     | 植中　朝日        | 横パス        | 0      | 9     
2          | 植中　朝日        | -2     | 井上　健太        | 横パス        | 0      | 10    
2          | 植中　朝日        | -1     | 加藤　聖         | 横パス        | 0      | 10    
------------------------------------------------------------------------------------------
8          | 植中　朝日        | -4     | 加藤　聖         | 縦パス        | 4      | 8     
8          | 植中　朝日        | -3     | 宮市　亮         | 横パス        | 0      | 9     
8          | 植中　朝日        | -2     | 植

In [5]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. 分析したいチーム名をここで指定してください
# ------------------------------------------------------------------
target_team_name ="横浜Ｆ・マリノス"  # ← ここを '川崎フロンターレ' や 'アルビレックス新潟' などに変更可能

# ------------------------------------------------------------------
# 2. データ準備 (play.csv読み込み & トラッキングデータ擬似生成)
# ------------------------------------------------------------------
try:
    # play.csv 読み込み
    raw_play = pd.read_csv('play.csv', encoding='utf-8-sig')

    # 列名変更 (日本語対応)
    col_map = {
        'チーム名': 'Team', 'アクション名': 'Action',
        '攻撃履歴No': 'AttackID', 'ボールＸ': 'x', 'ボールＹ': 'y',
        'ハーフ開始相対時間': 'Time', '履歴No': 'HistoryNo', '攻撃方向': 'Dir',
        '選手名': 'Player'
    }
    df = raw_play.rename(columns={k: v for k, v in col_map.items() if k in raw_play.columns})

    # 座標変換 & Frame生成
    df['x_m'] = pd.to_numeric(df['x'], errors='coerce') / 3.0
    df['y_m'] = pd.to_numeric(df['y'], errors='coerce') / 3.0
    time_col = df.get('Time', df.get('HistoryNo'))
    df['Frame'] = (time_col * 25 + 100000).astype(int)

    # トラッキングデータ(擬似)生成
    target_frames = df['Frame'].unique()
    track_records = []
    # ※本来は本物のトラッキングデータを使いますが、ここではデモ用に生成します
    for f in target_frames:
        for i in range(1, 12):
            track_records.append({'Frame': f, 'No': i, 'X': np.random.randint(-4000, 4000), 'Y': np.random.randint(-3000, 3000)})
    df_track = pd.DataFrame(track_records)

    # ------------------------------------------------------------------
    # 3. 個別分析ロジック実行
    # ------------------------------------------------------------------
    print(f"\n【分析対象チーム: {target_team_name}】\n")

    # チーム絞り込み
    team_df = df[df['Team'] == target_team_name].copy()
    if team_df.empty:
        print("エラー: 指定されたチームのデータが見つかりません。")
    else:
        # シュートイベント抽出
        shots = team_df[team_df['Action'].str.contains('シュート|ゴール', na=False)]
        print(f"分析対象シュート数: {len(shots)} シーン\n")

        print("-" * 90)
        print(f"{'AttackID':<10} | {'Shooter':<12} | {'Order':<6} | {'Passer':<12} | {'Type':<10} | {'V_Pack':<6} | {'D_Out':<6}")
        print("-" * 90)

        # 各シュートについてループ
        for idx, shot in shots.iterrows():
            attack_id = shot['AttackID']
            sequence = team_df[team_df['AttackID'] == attack_id].sort_values('Frame')
            try:
                shot_seq_idx = sequence.index.get_loc(idx)
            except: continue

            # シュートシーンごとのデータを格納するリスト
            scene_passes = []
            pass_count = 0

            # 遡ってパスを解析
            for i in range(1, 20):
                if shot_seq_idx - i < 0: break
                if pass_count >= 5: break

                event = sequence.iloc[shot_seq_idx - i]

                if 'パス' in str(event['Action']) or 'クロス' in str(event['Action']):
                    pass_count += 1

                    # パッキング計算
                    start_x = event['x_m']
                    next_event = sequence.iloc[shot_seq_idx - i + 1]
                    end_x = next_event['x_m']
                    if pd.isna(end_x): continue
                    if event['Dir'] == 2: start_x, end_x = -start_x, -end_x

                    frame = event['Frame']
                    opponents = df_track[df_track['Frame'] == frame]
                    opp_x = opponents['X'] / 100.0

                    v_packed = 0
                    if end_x > start_x:
                        v_packed = opponents[(opp_x > start_x) & (opp_x < end_x)].shape[0]
                    d_outplayed = opponents[opp_x < end_x].shape[0]

                    dx = end_x - start_x
                    dy = abs(next_event['y_m'] - event['y_m'])
                    p_type = 'Vertical' if (dx > 5 and dx > dy) else ('Horizontal' if (dy > dx and dy > 5) else 'Link-up')

                    # 結果をリストに追加
                    scene_passes.append({
                        'Order': -pass_count,
                        'Passer': event['Player'],
                        'Type': p_type,
                        'V_Pack': v_packed,
                        'D_Out': d_outplayed
                    })

            # 順番を整えて表示 (-5 -> -1)
            for p in sorted(scene_passes, key=lambda x: x['Order']):
                # パスの種類を日本語化して見やすく
                type_str = "縦パス" if p['Type'] == 'Vertical' else ("横パス" if p['Type'] == 'Horizontal' else "繋ぎ")

                print(f"{attack_id:<10} | {str(shot['Player']):<12} | {p['Order']:<6} | {str(p['Passer']):<12} | {type_str:<10} | {p['V_Pack']:<6} | {p['D_Out']:<6}")

            # シーンごとの区切り線
            if len(scene_passes) > 0:
                print("-" * 90)

except Exception as e:
    print(f"エラーが発生しました: {e}")


【分析対象チーム: 横浜Ｆ・マリノス】

分析対象シュート数: 9 シーン

------------------------------------------------------------------------------------------
AttackID   | Shooter      | Order  | Passer       | Type       | V_Pack | D_Out 
------------------------------------------------------------------------------------------
52         | 畠中　槙之輔       | -5     | 西村　拓真        | 横パス        | 0      | 11    
52         | 畠中　槙之輔       | -4     | ヤン　マテウス      | 繋ぎ         | 0      | 8     
52         | 畠中　槙之輔       | -3     | 永戸　勝也        | 横パス        | 0      | 10    
52         | 畠中　槙之輔       | -2     | 山根　陸         | 横パス        | 2      | 9     
52         | 畠中　槙之輔       | -1     | 渡邊　泰基        | 横パス        | 0      | 11    
------------------------------------------------------------------------------------------
52         | エウベル         | -5     | 永戸　勝也        | 横パス        | 0      | 10    
52         | エウベル         | -4     | 山根　陸         | 横パス        | 2      | 9     
52         | エウベル         | -3     | 渡邊